## MHAR Cleaning

**Purpose:** The purpose of this notebook is to generate the MHAR dataframe from the *IntradaySummaries_.STOXX50E_19980220_20200809.csv*. We compute the variables RVD, RVW, RVM, logRVD, logRVW, logRVM, RVD_pos, RVD_neg, RD_neg, RW_neg, RM_neg, and RQ. Note the variables RVD, RVW, RVM are part of the true MHAR dataset, whereas the remaining variables are used for the various extensions of the HAR model. 

- **RVD, RVW, RVM** — daily, weekly (5-day), and monthly (22-day) realized variance  
- **logRVD, logRVW, logRVM** — log-transformed versions of realized variance  
- **RVD_pos, RVD_neg** — positive and negative semivariance (for the SHAR model)  
- **RD_neg, RW_neg, RM_neg** — negative return aggregates (for the LevHAR model)  
- **RQ** — realized quarticity (for the HARQ model)

### Load CSV

In [3]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Raw Data from Tobias Sichert/IntradaySummaries_.STOXX50E_19980220_20200809.csv"
df = pd.read_csv(file_path) 
df.tail()

,#RIC,Alias Underlying RIC,Domain,Date-Time,GMT Offset,Type,Open,High,Low,Last,...,Close Discount Factor,No. Discount Factors,Open Bid Size,High Bid Size,Low Bid Size,Close Bid Size,Open Ask Size,High Ask Size,Low Ask Size,Close Ask Size
11817308,.STOXX50E,NaN,Market Price,2020-08-09T23:55:00.000000000Z,2,Intraday 1Min,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11817309,.STOXX50E,NaN,Market Price,2020-08-09T23:56:00.000000000Z,2,Intraday 1Min,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11817310,.STOXX50E,NaN,Market Price,2020-08-09T23:57:00.000000000Z,2,Intraday 1Min,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11817311,.STOXX50E,NaN,Market Price,2020-08-09T23:58:00.000000000Z,2,Intraday 1Min,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11817312,.STOXX50E,NaN,Market Price,2020-08-09T23:59:00.000000000Z,2,Intraday 1Min,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.columns

Index(['#RIC', 'Alias Underlying RIC', 'Domain', 'Date-Time', 'GMT Offset',
       'Type', 'Open', 'High', 'Low', 'Last', 'Volume', 'No. Trades',
       'Open Bid', 'High Bid', 'Low Bid', 'Close Bid', 'No. Bids', 'Open Ask',
       'High Ask', 'Low Ask', 'Close Ask', 'No. Asks', 'Open Yld', 'High Yld',
       'Low Yld', 'Close Yld', 'No. Ylds', 'Open Bid Yld', 'High Bid Yld',
       'Low Bid Yld', 'Close Bid Yld', 'No. Bid Ylds', 'Open Ask Yld',
       'High Ask Yld', 'Low Ask Yld', 'Close Ask Yld', 'No. Ask Ylds',
       'Open Zero Yld', 'High Zero Yld', 'Low Zero Yld', 'Close Zero Yld',
       'No. Zero Ylds', 'Open Discount Factor', 'High Discount Factor',
       'Low Discount Factor', 'Close Discount Factor', 'No. Discount Factors',
       'Open Bid Size', 'High Bid Size', 'Low Bid Size', 'Close Bid Size',
       'Open Ask Size', 'High Ask Size', 'Low Ask Size', 'Close Ask Size'],
      dtype='object')

### Converting "Date-Time" column into date time format and creating "Date-Time-CET" column

In [6]:
# Copy of data frame
df_copy = df.copy()
df_copy["Date-Time"] = pd.to_datetime(df_copy["Date-Time"], utc=True, errors="coerce")

# Convert UTC timestamps to CET/CEST (Europe/Berlin handles DST automatically)
df_copy["Date-Time-CET"] = df_copy["Date-Time"].dt.tz_convert("Europe/Berlin")

### Remove weekends

In [8]:
# Creating a "Weekday" column
df_copy["Weekday"] = df_copy["Date-Time-CET"].dt.weekday  # 0=Mon, 6=Sun

# Keep only weekdays (Monday=0 through Friday=4)
df_copy = df_copy[df_copy["Weekday"] <= 4].copy()

### Retrieving the correct trading window (09:00 - 17:30)¶

In [10]:
#  Creating a time stamp column
df_copy["Time_CET"] = df_copy["Date-Time-CET"].dt.time

# Trading window
start_time = pd.Timestamp("09:00").time()
end_time = pd.Timestamp("17:30").time()

# Filtering out non-trading hours 
df_copy = df_copy[
    (df_copy["Time_CET"] >= start_time) &
    (df_copy["Time_CET"] <= end_time)
].copy()

### Removing 1998-02-20 which is an incomplete first day 

In [12]:
# Remove the incomplete first day
df_copy = df_copy[df_copy["Date-Time-CET"].dt.date != pd.Timestamp("1998-02-20").date()].copy()

### Set Date-Time-CET column as index

In [14]:
df_copy = df_copy.sort_values("Date-Time-CET").set_index("Date-Time-CET")

### Forward-filling and removing days with >50% missing trading data

Removing public holidays as well as other dates with missing trading data. We proceed by forward-filling the values of those days that have a few number of bars missing

In [17]:
# Remove any day with more than 50% missing Last prices (>255 missing bars)
missing_per_day = df_copy["Last"].isna().groupby(df_copy.index.date).sum()
days_to_remove = missing_per_day[missing_per_day > 255].index

date_series = pd.Series(df_copy.index.date, index=df_copy.index)
df_copy = df_copy[~date_series.isin(days_to_remove)].copy()

print(f"Removed {len(days_to_remove)} days with >50% missing bars")
print(f"Remaining unique trading days: {df_copy.index.date.__len__() // 510}")

# Forward fill remaining missing values within each trading day
df_copy["Last_filled"] = (
    df_copy["Last"]
    .groupby(df_copy.index.date)
    .transform(lambda x: x.ffill().bfill())
)

print(f"Remaining NaN after forward fill: {df_copy['Last_filled'].isna().sum()}")

Removed 164 days with >50% missing bars
Remaining unique trading days: 5707
Remaining NaN after forward fill: 0


### Computing log returns

Let $P_{t,i}$ the EURO STOXX 50 index level at intraday minute $i$ on trading day $t$
We define the 1-minute log return as: 
$$
r_{t,i} = \log\!\left(P_{t,i}\right) - \log\!\left(P_{t,i-1}\right)
$$

where: 
* $i = 1,..., M_t$
* $M_t$ is the number of intraday observations during trading hours (09:00-17:30 CET). Note that returns are computed within each trading day only, so overnight returns are exluded at this stage. 

In [20]:
# Step 1: Create a date column for grouping
df_copy["Date"] = df_copy.index.date

# Step 2: Compute log price
df_copy["log_last"] = np.log(df_copy["Last_filled"])

# Step 3: Compute 1-minute log returns within each day only
# .diff() will return NaN for the first bar of each day, which is correct
df_copy["r_1m"] = (
    df_copy["log_last"]
    .groupby(df_copy["Date"])
    .diff()
)

# Removing corrupt day with data error

In [22]:
# Remove the corrupted day
date_to_remove = pd.Timestamp("1998-06-29").date()
date_series = pd.Series(df_copy.index.date, index=df_copy.index)
df_copy = df_copy[~date_series.isin([date_to_remove])].copy()

# Verify removal
print(f"Remaining unique trading days: {len(np.unique(df_copy.index.date))}")

# Recompute returns after removal
df_copy["log_last"] = np.log(df_copy["Last_filled"])
df_copy["r_1m"] = (
    df_copy["log_last"]
    .groupby(df_copy["Date"])
    .diff()
)

# Check for extreme returns
print("\nUpdated return distribution:")
print(df_copy["r_1m"].describe())
print("\nInfinite returns:", np.isinf(df_copy["r_1m"]).sum())
print("Returns > 10% in absolute value:", (df_copy["r_1m"].abs() > 0.10).sum())

Remaining unique trading days: 5695

Updated return distribution:
count    2.904450e+06
mean    -4.772466e-07
std      4.884938e-04
min     -4.801300e-02
25%     -1.731069e-04
50%      0.000000e+00
75%      1.771284e-04
max      3.799486e-02
Name: r_1m, dtype: float64

Infinite returns: 0
Returns > 10% in absolute value: 0


### Intraday realized variance 

Intraday realized variance for day $t$ is defined as: 

$$
RV_t^{\text{intra}} = \sum_{i=1}^{M_t} r_{t,i}^2
$$

This measure captures the total variation accumulated during regular trading hours

In [37]:
# Compute intraday RV as sum of squared 1-minute log returns within each day
# Note: NaN returns (first bar of each day) are automatically excluded by sum()
RV_intraday = (
    df_copy["r_1m"]
    .pow(2)
    .groupby(df_copy["Date"])
    .sum()
)

RV_intraday.name = "RV_intraday"

print(RV_intraday.describe())
print(f"\nDate range: {RV_intraday.index[0]} to {RV_intraday.index[-1]}")
print(f"Number of trading days: {len(RV_intraday)}")

count    5.695000e+03
mean     1.216994e-04
std      2.090949e-04
min      8.284016e-09
25%      3.632089e-05
50%      6.666649e-05
75%      1.279737e-04
max      5.199574e-03
Name: RV_intraday, dtype: float64

Date range: 1998-02-26 to 2020-08-07
Number of trading days: 5695


### Overnight realized variance

The following equation is used to compute overnight realized variance
$$
RV_t^{ON} = (r_t^{ON})^2 = \left(\ln \frac{P_{t+1}^{\text{open}}}{P_t^{\text{close}}}\right)^2
$$

In [26]:
# Get the last price of each trading day (17:30 bar)
last_price_of_day = (
    df_copy["Last_filled"]
    .groupby(df_copy["Date"])
    .last()
)

# Get the first price of each trading day (09:00 bar)
first_price_of_day = (
    df_copy["Last_filled"]
    .groupby(df_copy["Date"])
    .first()
)

# Compute overnight log return as log(open today / close yesterday)
# shift(1) aligns yesterday's closing price with today's date
overnight_return = np.log(first_price_of_day / last_price_of_day.shift(1))
overnight_return.name = "overnight_return"

# Square to get overnight variance contribution
RV_overnight = overnight_return.pow(2)
RV_overnight.name = "RV_overnight"

### Total daily realized variance

Daily realized variance including overnight return is computed as:

$$
RV_t^{\text{total}} = RV_t^{\text{intra}} + \left(r_t^{\text{overnight}}\right)^2
$$

This corresponds to close-to-close return variation and is directly comparable to implied variance measures such as the VSTOXX and VIX.

In [29]:
# Combine intraday and overnight RV into total daily RV
RV_total = pd.DataFrame({
    "RV_intraday": RV_intraday,
    "RV_overnight": RV_overnight
})

# Total RV = intraday RV + overnight RV
RV_total["RVD"] = RV_total["RV_intraday"] + RV_total["RV_overnight"]

# The first day will have NaN due to missing overnight return
# Drop it to keep a clean series
RV_total = RV_total.dropna()

### Computing weekly and monthly realized variances and their corresponding log versions

In [31]:
# Start fresh from RV_total with RVD, RV_intraday and RV_overnight
# Step 1: RVW and RVM
RV_total["RVW"] = RV_total["RVD"].rolling(window=5).mean()
RV_total["RVM"] = RV_total["RVD"].rolling(window=22).mean()

# Step 2: Log versions
RV_total["logRVD"] = np.log(RV_total["RVD"])
RV_total["logRVW"] = np.log(RV_total["RVW"])
RV_total["logRVM"] = np.log(RV_total["RVM"])

### Computing positive and negative realized variances for Semivariance HAR (SHAR)

Computing RVD_pos and RVD_neg

In [34]:
# Compute positive and negative semivariance (intraday only)
RV_total["RVD_pos"] = (
    df_copy["r_1m"]
    .where(df_copy["r_1m"] > 0, 0)
    .pow(2)
    .groupby(df_copy["Date"])
    .sum()
)

RV_total["RVD_neg"] = (
    df_copy["r_1m"]
    .where(df_copy["r_1m"] < 0, 0)
    .pow(2)
    .groupby(df_copy["Date"])
    .sum()
)

# Allocate overnight return to pos or neg semivariance based on its sign
overnight_pos = RV_total["RV_overnight"].where(overnight_return > 0, 0)
overnight_neg = RV_total["RV_overnight"].where(overnight_return < 0, 0)

# Add overnight component to intraday semivariance
RV_total["RVD_pos"] = RV_total["RVD_pos"] + overnight_pos
RV_total["RVD_neg"] = RV_total["RVD_neg"] + overnight_neg

### Computing cumulative negative return on daily, weekly, and monthly basis for LevHAR model

Computing RD_neg, RW_neg, RM_neg

In [37]:
# Compute daily close-to-close LOG returns
daily_close = df_copy["Last_filled"].groupby(df_copy["Date"]).last()
daily_log_return = np.log(daily_close / daily_close.shift(1))

# Negative portion: min(0, r_t)
daily_log_return_neg = daily_log_return.clip(upper=0)

# Rolling averages of negative log returns
RV_total["RD_neg"] = daily_log_return_neg
RV_total["RW_neg"] = daily_log_return_neg.rolling(window=5).mean()
RV_total["RM_neg"] = daily_log_return_neg.rolling(window=22).mean()

### Computing realized quarticity for the HARQ model

For intraday returns $r_{t,i}$ realized quarticity is defined as: 
$$
RQ_t = \frac{M_t}{3} \sum_{i=1}^{M_t} r_{t,i}^4
$$

where: 
* $M_t$ = umber of intraday returns on day $t$
* The caling factor $\frac{M_t}{3}$ ensures consistency under continuous semimartingale assumptions

RQ estimates the integrated quarticity ($IQ$), which captures the variability of volatility itself 

It is used in HARQ to scale the impacy of lagged realized variance 

In [40]:
# Use correct n for your sampling frequency
n = 509  # number of 1-minute returns per day (510 bars minus first NaN)

# Recompute RQ with correct scaling
RV_total["RQ"] = (
    df_copy["r_1m"]
    .pow(4)
    .groupby(df_copy["Date"])
    .sum()
    .mul(n / 3)
)

### Complete MHAR data set

In [42]:
# Create final dataframe with all required variables
MHAR = RV_total[[
    "RVD", "RVW", "RVM",
    "logRVD", "logRVW", "logRVM",
    "RVD_pos", "RVD_neg",
    "RD_neg", "RW_neg", "RM_neg",
    "RQ"
]].copy()

# Drop NaN values introduced by rolling windows
MHAR = MHAR.dropna()

In [43]:
from IPython.display import display
import pandas as pd
import numpy as np

# Prepare display variables
MHAR_display = pd.DataFrame({
    "RVD":     np.sqrt(MHAR["RVD"] * 252) * 100,
    "RVW":     np.sqrt(MHAR["RVW"] * 252) * 100,
    "RVM":     np.sqrt(MHAR["RVM"] * 252) * 100,
    "logRVD":  MHAR["logRVD"],
    "logRVW":  MHAR["logRVW"],
    "logRVM":  MHAR["logRVM"],
    "RVD_pos": np.sqrt(MHAR["RVD_pos"] * 252) * 100,
    "RVD_neg": np.sqrt(MHAR["RVD_neg"] * 252) * 100,
    "RD_neg":  MHAR["RD_neg"] * 100,
    "RW_neg":  MHAR["RW_neg"] * 100,
    "RM_neg":  MHAR["RM_neg"] * 100,
    "RQ":      MHAR["RQ"] * 1e6,
})

# Compute summary statistics as rows
summary = pd.DataFrame({
    "Observations":  MHAR_display.count(),
    "Mean":          MHAR_display.mean(),
    "Median":        MHAR_display.median(),
    "Std. Dev.":     MHAR_display.std(),
    "Skewness":      MHAR_display.skew(),
    "Exc. Kurtosis": MHAR_display.kurt(),
    "Minimum":       MHAR_display.min(),
    "Maximum":       MHAR_display.max(),
}).T.round(4)

# Format each row appropriately
def format_value(val, row_name):
    if row_name == "Observations":
        return f"{int(val):,}"
    else:
        return f"{val:,.4f}"

# Style the table
styled_summary = (
    summary.style
    .set_caption(
        "Table 1: Summary Statistics of MHAR Feature Set — EURO STOXX 50 "
        f"({MHAR.index[0].strftime('%Y-%m-%d')} to {MHAR.index[-1].strftime('%Y-%m-%d')}). "
        "RVD, RVW, RVM, RVD$_{pos}$, RVD$_{neg}$ are expressed as annualized std (%). "
        "RD$_{neg}$, RW$_{neg}$, RM$_{neg}$ are expressed as %. "
        "RQ is scaled by $10^{6}$."
    )
    .format(lambda x: f"{x:,.4f}")
    .format(lambda x: f"{int(x):,}", subset=pd.IndexSlice["Observations", :])
    .set_table_styles([
        # Caption
        {"selector": "caption",
         "props": [
             ("font-size", "12px"),
             ("font-weight", "bold"),
             ("text-align", "left"),
             ("padding-bottom", "8px"),
             ("color", "black"),
             ("caption-side", "top"),
         ]},
        # Header - variable names
        {"selector": "thead tr th",
         "props": [
             ("background-color", "white"),
             ("color", "black"),
             ("font-size", "11px"),
             ("text-align", "center"),
             ("padding", "6px 10px"),
             ("border-top", "2px solid black"),
             ("border-bottom", "1px solid black"),
             ("font-weight", "bold"),
         ]},
        # Index - statistic names
        {"selector": "tbody tr th",
         "props": [
             ("font-size", "11px"),
             ("font-weight", "normal"),
             ("text-align", "left"),
             ("padding", "4px 10px"),
             ("color", "black"),
             ("background-color", "white"),
             ("border-right", "1px solid black"),
         ]},
        # Cells
        {"selector": "tbody tr td",
         "props": [
             ("font-size", "11px"),
             ("text-align", "right"),
             ("padding", "4px 10px"),
             ("background-color", "white"),
             ("color", "black"),
         ]},
        # Last row bottom border
        {"selector": "tbody tr:last-child td, tbody tr:last-child th",
         "props": [("border-bottom", "2px solid black")]},
        # Table
        {"selector": "table",
         "props": [
             ("border-collapse", "collapse"),
             ("width", "100%"),
             ("border-top", "2px solid black"),
             ("border-bottom", "2px solid black"),
         ]},
        # Remove default borders
        {"selector": "td, th",
         "props": [
             ("border-left", "none"),
             ("border-right", "none"),
         ]},
    ])
)

display(styled_summary)

,RVD,RVW,RVM,logRVD,logRVW,logRVM,RVD_pos,RVD_neg,RD_neg,RW_neg,RM_neg,RQ
Observations,"5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673"
Mean,16.9550,17.3447,17.6268,-9.3338,-9.2422,-9.1833,11.8350,11.7627,-0.5034,-0.5035,-0.5030,0.5039
Median,14.5408,15.0991,15.5698,-9.3858,-9.3105,-9.2491,10.0662,9.8351,0.0000,-0.3620,-0.4119,0.0117
Std. Dev.,9.7885,9.0812,8.5124,0.9842,0.8778,0.8193,6.9304,7.5385,0.9268,0.5006,0.3548,13.7511
Skewness,2.4817,2.2708,1.9918,0.2512,0.5568,0.6248,2.4689,2.7064,-3.2568,-2.3954,-2.0569,56.5281
Exc. Kurtosis,10.7050,8.1185,5.5477,0.8157,0.2917,0.2080,10.2918,12.9967,16.2600,10.1550,6.0504,"3,566.8231"
Minimum,0.7844,4.5097,5.8217,-15.2255,-11.7273,-11.2166,0.7506,0.1122,-12.0054,-5.4131,-2.7391,0.0000
Maximum,114.5369,82.3138,68.5179,-5.2580,-5.9187,-6.2856,71.2714,95.6159,0.0000,0.0000,-0.0538,915.9163


In [43]:
from IPython.display import display
import pandas as pd
import numpy as np

# Prepare display variables
MHAR_display = pd.DataFrame({
    "RVD":     np.sqrt(MHAR["RVD"] * 252) * 100,
    "RVW":     np.sqrt(MHAR["RVW"] * 252) * 100,
    "RVM":     np.sqrt(MHAR["RVM"] * 252) * 100,
    "logRVD":  MHAR["logRVD"],
    "logRVW":  MHAR["logRVW"],
    "logRVM":  MHAR["logRVM"],
    "RVD_pos": np.sqrt(MHAR["RVD_pos"] * 252) * 100,
    "RVD_neg": np.sqrt(MHAR["RVD_neg"] * 252) * 100,
    "RD_neg":  MHAR["RD_neg"] * 100,
    "RW_neg":  MHAR["RW_neg"] * 100,
    "RM_neg":  MHAR["RM_neg"] * 100,
    "RQ":      MHAR["RQ"] * 1e6,
})

# Compute summary statistics as rows
summary = pd.DataFrame({
    "Observations":  MHAR_display.count(),
    "Mean":          MHAR_display.mean(),
    "Median":        MHAR_display.median(),
    "Std. Dev.":     MHAR_display.std(),
    "Skewness":      MHAR_display.skew(),
    "Exc. Kurtosis": MHAR_display.kurt(),
    "Minimum":       MHAR_display.min(),
    "Maximum":       MHAR_display.max(),
}).T.round(4)

# Format each row appropriately
def format_value(val, row_name):
    if row_name == "Observations":
        return f"{int(val):,}"
    else:
        return f"{val:,.4f}"

# Style the table
styled_summary = (
    summary.style
    .set_caption(
        "Table 1: Summary Statistics of MHAR Feature Set — EURO STOXX 50 "
        f"({MHAR.index[0].strftime('%Y-%m-%d')} to {MHAR.index[-1].strftime('%Y-%m-%d')}). "
        "RVD, RVW, RVM, RVD$_{pos}$, RVD$_{neg}$ are expressed as annualized std (%). "
        "RD$_{neg}$, RW$_{neg}$, RM$_{neg}$ are expressed as %. "
        "RQ is scaled by $10^{6}$."
    )
    .format(lambda x: f"{x:,.4f}")
    .format(lambda x: f"{int(x):,}", subset=pd.IndexSlice["Observations", :])
    .set_table_styles([
        # Caption
        {"selector": "caption",
         "props": [
             ("font-size", "12px"),
             ("font-weight", "bold"),
             ("text-align", "left"),
             ("padding-bottom", "8px"),
             ("color", "black"),
             ("caption-side", "top"),
         ]},
        # Header - variable names
        {"selector": "thead tr th",
         "props": [
             ("background-color", "white"),
             ("color", "black"),
             ("font-size", "11px"),
             ("text-align", "center"),
             ("padding", "6px 10px"),
             ("border-top", "2px solid black"),
             ("border-bottom", "1px solid black"),
             ("font-weight", "bold"),
         ]},
        # Index - statistic names
        {"selector": "tbody tr th",
         "props": [
             ("font-size", "11px"),
             ("font-weight", "normal"),
             ("text-align", "left"),
             ("padding", "4px 10px"),
             ("color", "black"),
             ("background-color", "white"),
             ("border-right", "1px solid black"),
         ]},
        # Cells
        {"selector": "tbody tr td",
         "props": [
             ("font-size", "11px"),
             ("text-align", "right"),
             ("padding", "4px 10px"),
             ("background-color", "white"),
             ("color", "black"),
         ]},
        # Last row bottom border
        {"selector": "tbody tr:last-child td, tbody tr:last-child th",
         "props": [("border-bottom", "2px solid black")]},
        # Table
        {"selector": "table",
         "props": [
             ("border-collapse", "collapse"),
             ("width", "100%"),
             ("border-top", "2px solid black"),
             ("border-bottom", "2px solid black"),
         ]},
        # Remove default borders
        {"selector": "td, th",
         "props": [
             ("border-left", "none"),
             ("border-right", "none"),
         ]},
    ])
)

display(styled_summary)

,RVD,RVW,RVM,logRVD,logRVW,logRVM,RVD_pos,RVD_neg,RD_neg,RW_neg,RM_neg,RQ
Observations,"5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673","5,673"
Mean,16.9550,17.3447,17.6268,-9.3338,-9.2422,-9.1833,11.8350,11.7627,-0.5034,-0.5035,-0.5030,0.5039
Median,14.5408,15.0991,15.5698,-9.3858,-9.3105,-9.2491,10.0662,9.8351,0.0000,-0.3620,-0.4119,0.0117
Std. Dev.,9.7885,9.0812,8.5124,0.9842,0.8778,0.8193,6.9304,7.5385,0.9268,0.5006,0.3548,13.7511
Skewness,2.4817,2.2708,1.9918,0.2512,0.5568,0.6248,2.4689,2.7064,-3.2568,-2.3954,-2.0569,56.5281
Exc. Kurtosis,10.7050,8.1185,5.5477,0.8157,0.2917,0.2080,10.2918,12.9967,16.2600,10.1550,6.0504,"3,566.8231"
Minimum,0.7844,4.5097,5.8217,-15.2255,-11.7273,-11.2166,0.7506,0.1122,-12.0054,-5.4131,-2.7391,0.0000
Maximum,114.5369,82.3138,68.5179,-5.2580,-5.9187,-6.2856,71.2714,95.6159,0.0000,0.0000,-0.0538,915.9163


### Saving the completed data frame in CSV format: MHAR.csv

In [45]:
# Save MHAR dataframe to CSV
save_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data Sets/MHAR.csv"
MHAR.to_csv(save_path)

# Verify it saved correctly
MHAR_check = pd.read_csv(save_path, index_col="Date", parse_dates=True)
print("Saved and reloaded successfully:")
print(f"Shape: {MHAR_check.shape}")
print(f"Date range: {MHAR_check.index[0].strftime('%Y-%m-%d')} to {MHAR_check.index[-1].strftime('%Y-%m-%d')}")
print(f"Columns: {MHAR_check.columns.tolist()}")
print(f"NaN values: {MHAR_check.isna().sum().sum()}")

Saved and reloaded successfully:
Shape: (5673, 12)
Date range: 1998-03-31 to 2020-08-07
Columns: ['RVD', 'RVW', 'RVM', 'logRVD', 'logRVW', 'logRVM', 'RVD_pos', 'RVD_neg', 'RD_neg', 'RW_neg', 'RM_neg', 'RQ']
NaN values: 0


#### For Momentum variable in MALL feature set 

In [47]:
# Saving daily closing prices 
daily_close = df_copy["Last_filled"].groupby(df_copy["Date"]).last()
daily_close.name = "Close"
daily_close.to_csv(
    "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data Sets/daily_close.csv",
    header=True
)
print(f"Saved {len(daily_close)} daily closing prices")
print(daily_close.head())

Saved 5695 daily closing prices
Date
1998-02-26    2874.81
1998-02-27    2878.19
1998-03-02    2929.86
1998-03-03    2909.37
1998-03-04    2876.53
Name: Close, dtype: float64
